# Build Augmented Matrix Simulation

จำลองการทำงานของฟังก์ชัน `build_augmented` จาก C++  
สร้าง augmented matrix **[A | b]** จาก `LinearForm` หลายตัว โดย:

- `A[i][j]` = สัมประสิทธิ์ของตัวแปร `var_order[j]` ใน form ที่ `i`
- `b[i]` = `-forms[i].constant`

In [1]:
from __future__ import annotations


class LinearForm:
    """
    แทน linear expression เช่น 2x + 3y - 5
    เก็บเป็น dict ของ {ชื่อตัวแปร: สัมประสิทธิ์} + ค่าคงที่
    """

    def __init__(self, coeffs: dict[str, float], constant: float = 0.0):
        # coeffs เช่น {"x": 2.0, "y": 3.0}
        self.coeffs = dict(coeffs)
        # ค่าคงที่ เช่น -5.0
        self.constant = constant

    def get_coeff(self, var: str) -> float:
        """คืนสัมประสิทธิ์ของตัวแปร var (ถ้าไม่มีคืน 0)"""
        return self.coeffs.get(var, 0.0)

    def __repr__(self) -> str:
        terms = [f"{c:+.4g}·{v}" for v, c in self.coeffs.items()]
        return f"LinearForm({' '.join(terms)} {self.constant:+.4g})"

In [2]:
def build_augmented(
    forms: list[LinearForm],
    var_order: list[str],
    verbose: bool = True,
) -> list[list[float]]:
    """
    สร้าง augmented matrix [A | b]

    Parameters
    ----------
    forms : list[LinearForm]
        สมการเชิงเส้นแต่ละตัว (m ตัว)
    var_order : list[str]
        ลำดับตัวแปรที่จะเรียงเป็นคอลัมน์ (n ตัว)
    verbose : bool
        ถ้า True จะ print debug ทุก loop ย่อย

    Returns
    -------
    list[list[float]]
        เมทริกซ์ขนาด m × (n+1)
    """
    m = len(forms)       # จำนวนแถว (สมการ)
    n = len(var_order)   # จำนวนตัวแปร

    if verbose:
        print(f"m = {m} forms,  n = {n} vars")
        print(f"var_order = {var_order}")
        # หัวคอลัมน์สำหรับ debug
        header = "  ".join(f"{v:>8}" for v in var_order) + "  |  " + f"{'b':>8}"
        print(f"\n{'':>6}{header}")
        print(f"{'':>6}{'-' * len(header)}")

    # สร้างเมทริกซ์ศูนย์ขนาด m × (n+1)
    mat = [[0.0] * (n + 1) for _ in range(m)]

    for i in range(m):
        if verbose:
            print(f"\n--- row {i}: {forms[i]} ---")

        # === inner loop: ดึงสัมประสิทธิ์ตัวแปรแต่ละตัว ===
        for j in range(n):
            coeff = forms[i].get_coeff(var_order[j])
            mat[i][j] = coeff

            if verbose:
                print(f"    A[{i}][{j}] = coeff('{var_order[j]}') = {coeff:>10.4g}")

        # === คอลัมน์สุดท้าย: b = -constant ===
        mat[i][n] = -forms[i].constant

        if verbose:
            print(f"    b[{i}]    = -constant = -({forms[i].constant}) = {mat[i][n]:>10.4g}")
            # แสดงแถวที่สร้างเสร็จ
            row_str = "  ".join(f"{v:>8.4g}" for v in mat[i][:n])
            print(f"    => [ {row_str}  | {mat[i][n]:>8.4g} ]")

    if verbose:
        print("\n===== Augmented Matrix [A | b] =====")
        for i, row in enumerate(mat):
            a_part = "  ".join(f"{v:>8.4g}" for v in row[:n])
            print(f"  row {i}: [ {a_part}  | {row[n]:>8.4g} ]")

    return mat

## Test Case 1: ระบบสมการ 2 ตัวแปร

```
2x + 3y = 7     →  2x + 3y - 7 = 0   (coeffs={x:2, y:3}, constant=-7)
 x -  y = 1     →   x -  y - 1 = 0   (coeffs={x:1, y:-1}, constant=-1)
```

augmented matrix ที่คาดหวัง:
```
[ 2   3  |  7 ]
[ 1  -1  |  1 ]
```

In [3]:
forms1 = [
    LinearForm({"x": 2.0, "y": 3.0}, constant=-7.0),
    LinearForm({"x": 1.0, "y": -1.0}, constant=-1.0),
]
var_order1 = ["x", "y"]

mat1 = build_augmented(forms1, var_order1)

assert mat1 == [
    [2.0,  3.0, 7.0],
    [1.0, -1.0, 1.0],
], f"Unexpected: {mat1}"

m = 2 forms,  n = 2 vars
var_order = ['x', 'y']

             x         y  |         b
      -------------------------------

--- row 0: LinearForm(+2·x +3·y -7) ---
    A[0][0] = coeff('x') =          2
    A[0][1] = coeff('y') =          3
    b[0]    = -constant = -(-7.0) =          7
    => [        2         3  |        7 ]

--- row 1: LinearForm(+1·x -1·y -1) ---
    A[1][0] = coeff('x') =          1
    A[1][1] = coeff('y') =         -1
    b[1]    = -constant = -(-1.0) =          1
    => [        1        -1  |        1 ]

===== Augmented Matrix [A | b] =====
  row 0: [        2         3  |        7 ]
  row 1: [        1        -1  |        1 ]


## Test Case 2: ตัวแปรบางตัวไม่ปรากฏในบาง form

```
x + z = 4       →  coeffs={x:1, z:1}, constant=-4
y - 2z = -1     →  coeffs={y:1, z:-2}, constant=1
```

var_order = [x, y, z] → form แรกไม่มี y (ได้ 0), form สองไม่มี x (ได้ 0)

```
[ 1   0   1  |  4 ]
[ 0   1  -2  | -1 ]
```

In [4]:
forms2 = [
    LinearForm({"x": 1.0, "z": 1.0}, constant=-4.0),
    LinearForm({"y": 1.0, "z": -2.0}, constant=1.0),
]
var_order2 = ["x", "y", "z"]

mat2 = build_augmented(forms2, var_order2)

assert mat2 == [
    [1.0, 0.0,  1.0,  4.0],
    [0.0, 1.0, -2.0, -1.0],
], f"Unexpected: {mat2}"

m = 2 forms,  n = 3 vars
var_order = ['x', 'y', 'z']

             x         y         z  |         b
      -----------------------------------------

--- row 0: LinearForm(+1·x +1·z -4) ---
    A[0][0] = coeff('x') =          1
    A[0][1] = coeff('y') =          0
    A[0][2] = coeff('z') =          1
    b[0]    = -constant = -(-4.0) =          4
    => [        1         0         1  |        4 ]

--- row 1: LinearForm(+1·y -2·z +1) ---
    A[1][0] = coeff('x') =          0
    A[1][1] = coeff('y') =          1
    A[1][2] = coeff('z') =         -2
    b[1]    = -constant = -(1.0) =         -1
    => [        0         1        -2  |       -1 ]

===== Augmented Matrix [A | b] =====
  row 0: [        1         0         1  |        4 ]
  row 1: [        0         1        -2  |       -1 ]


## Test Case 3: var_order เรียงลำดับต่างกัน

ใช้ form เดียวกับ Test 1 แต่สลับลำดับตัวแปรเป็น [y, x]  
คอลัมน์ควรสลับตาม:

```
[ 3   2  |  7 ]
[-1   1  |  1 ]
```

In [5]:
# ใช้ forms เดียวกับ Test 1 แต่สลับ var_order
var_order3 = ["y", "x"]

mat3 = build_augmented(forms1, var_order3)

assert mat3 == [
    [ 3.0, 2.0, 7.0],
    [-1.0, 1.0, 1.0],
], f"Unexpected: {mat3}"

m = 2 forms,  n = 2 vars
var_order = ['y', 'x']

             y         x  |         b
      -------------------------------

--- row 0: LinearForm(+2·x +3·y -7) ---
    A[0][0] = coeff('y') =          3
    A[0][1] = coeff('x') =          2
    b[0]    = -constant = -(-7.0) =          7
    => [        3         2  |        7 ]

--- row 1: LinearForm(+1·x -1·y -1) ---
    A[1][0] = coeff('y') =         -1
    A[1][1] = coeff('x') =          1
    b[1]    = -constant = -(-1.0) =          1
    => [       -1         1  |        1 ]

===== Augmented Matrix [A | b] =====
  row 0: [        3         2  |        7 ]
  row 1: [       -1         1  |        1 ]


## Test Case 4: ค่าคงที่เป็น 0 / สัมประสิทธิ์เป็นทศนิยม

```
0.5a - 1.5b = 0     →  coeffs={a:0.5, b:-1.5}, constant=0
```

```
[ 0.5  -1.5  |  0 ]
```

In [6]:
forms4 = [
    LinearForm({"a": 0.5, "b": -1.5}, constant=0.0),
]
var_order4 = ["a", "b"]

mat4 = build_augmented(forms4, var_order4)

assert mat4 == [
    [0.5, -1.5, 0.0],
], f"Unexpected: {mat4}"

m = 1 forms,  n = 2 vars
var_order = ['a', 'b']

             a         b  |         b
      -------------------------------

--- row 0: LinearForm(+0.5·a -1.5·b +0) ---
    A[0][0] = coeff('a') =        0.5
    A[0][1] = coeff('b') =       -1.5
    b[0]    = -constant = -(0.0) =         -0
    => [      0.5      -1.5  |       -0 ]

===== Augmented Matrix [A | b] =====
  row 0: [      0.5      -1.5  |       -0 ]


## Test Case 5: var_order มีตัวแปรที่ไม่มีใน form ใดเลย

```
3x = 6     →  coeffs={x:3}, constant=-6
```

var_order = [x, y, z] → y, z ไม่ปรากฏเลย → ได้ 0 หมด

```
[ 3   0   0  |  6 ]
```

In [7]:
forms5 = [
    LinearForm({"x": 3.0}, constant=-6.0),
]
var_order5 = ["x", "y", "z"]

mat5 = build_augmented(forms5, var_order5)

assert mat5 == [
    [3.0, 0.0, 0.0, 6.0],
], f"Unexpected: {mat5}"

m = 1 forms,  n = 3 vars
var_order = ['x', 'y', 'z']

             x         y         z  |         b
      -----------------------------------------

--- row 0: LinearForm(+3·x -6) ---
    A[0][0] = coeff('x') =          3
    A[0][1] = coeff('y') =          0
    A[0][2] = coeff('z') =          0
    b[0]    = -constant = -(-6.0) =          6
    => [        3         0         0  |        6 ]

===== Augmented Matrix [A | b] =====
  row 0: [        3         0         0  |        6 ]


## Test Case 6: เมทริกซ์ว่าง

In [ ]:
# ไม่มี form เลย → เมทริกซ์ 0 แถว
print("=== no forms ===")
mat6a = build_augmented([], ["x", "y"])
assert mat6a == []

print()

# ไม่มีตัวแปร → เหลือแค่คอลัมน์ b
print("=== no variables ===")
forms6b = [LinearForm({}, constant=-5.0)]
mat6b = build_augmented(forms6b, [])
assert mat6b == [[5.0]], f"Unexpected: {mat6b}"

## สรุป

| Test | สถานการณ์ | ขนาด m×(n+1) | ตรวจอะไร |
|------|-----------|-------------|----------|
| 1 | 2 สมการ 2 ตัวแปร ปกติ | 2×3 | ค่าสัมประสิทธิ์ + b ถูกต้อง |
| 2 | ตัวแปรบางตัวหายไป | 2×4 | `get_coeff` คืน 0 ถ้าไม่มี |
| 3 | สลับ var_order | 2×3 | คอลัมน์เรียงตาม var_order |
| 4 | ค่าทศนิยม + constant=0 | 1×3 | b = -0 = 0 |
| 5 | ตัวแปรเกินใน var_order | 1×4 | คอลัมน์เกินเป็น 0 |
| 6 | edge cases ว่าง | 0×?, 1×1 | ไม่พัง |